In [22]:
import pandas as pd
import jieba
import logging
from gensim.models import Word2Vec

# 关闭冗余日志，只显示关键结果
logging.basicConfig(level=logging.ERROR)
print("✅ 库导入完成，日志已关闭，开始处理数据")

✅ 库导入完成，日志已关闭，开始处理数据


In [23]:
# 加载train.csv数据集
df = pd.read_csv("train.csv", encoding="utf-8")
# 清洗数据：去除空评论和空白评论
df = df.dropna(subset=["comment"])
df = df[df["comment"].str.strip() != ""]
# 结巴分词，生成Word2Vec所需语料（列表嵌套列表）
df["cut_comment"] = df["comment"].apply(lambda x: list(jieba.cut(x.strip())))
corpus = df["cut_comment"].tolist()

print(f"✅ 数据处理完成")
print(f"有效评论数：{len(corpus)}")
print(f"第一条分词结果：{corpus[0]}")

✅ 数据处理完成
有效评论数：10000
第一条分词结果：['一如既往', '地', '好吃', '，', '希望', '可以', '开', '到', '其他', '城市']


In [24]:
# 训练Skip-Gram模型（sg=1 代表Skip-Gram，必须设置为1）
w2v_model = Word2Vec(
    sentences=corpus,
    sg=1,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    epochs=10,
    seed=2025
)
# 保存模型
w2v_model.save("word2vec_skipgram.model")

print("✅ 子任务1完成：Skip-Gram模型训练并保存成功！")
print(f"模型词汇表大小：{len(w2v_model.wv)} 个词")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


✅ 子任务1完成：Skip-Gram模型训练并保存成功！
模型词汇表大小：2777 个词


In [25]:
word = "环境"
if word in w2v_model.wv:
    vec = w2v_model.wv[word]
    vec_shape = vec.shape
    print(f"✅ 子任务2完成：“{word}”的词向量及形状")
    print(f"词向量：\n{vec}")
    print(f"向量形状：{vec_shape}（{vec_shape[0]}维稠密向量）")
else:
    print(f"⚠️  语料中无“{word}”一词，可降低min_count重试")

✅ 子任务2完成：“环境”的词向量及形状
词向量：
[ 0.42264813 -0.01256103 -0.15675981  0.06842504  0.08398286  0.68697184
  0.72581303  0.24339864  0.10928377 -0.04166795  0.4469527   0.11710507
 -0.3672445   0.63646317  0.32685378 -0.22249591  0.14037164  0.60691696
  0.23534493 -0.14490989 -0.09212194 -0.58402985 -0.5930388  -0.07433119
 -0.03998051 -0.23151194  0.3261272   0.3673091   0.8504516   0.25049862
 -0.00902406  0.03276878 -0.15988512 -0.36920422  0.15857407 -0.01139596
  0.1869814   0.248724    0.3352872  -0.29703072 -0.5406334   0.27733675
 -0.43159863 -0.21681233 -0.41002613  0.18988511 -0.6304888  -0.21751755
  0.45631582 -0.01378668 -0.1148505  -0.01166258 -0.368371   -0.6989504
 -0.02685831  0.37739336 -0.14718904 -0.14997959 -0.1154372   0.23982401
 -0.00999833  0.36753282  0.33795434  0.44755083 -0.2141608   0.38187996
  0.13181314  0.7074229  -0.13192269  0.13895415  0.3210138   0.07448653
  0.34541973 -0.25531814  0.22309268  0.26616344 -0.15955293  0.1730082
  0.11238628  0.14023325 -0

In [26]:
target_word = "好吃"
if target_word in w2v_model.wv:
    similar_words = w2v_model.wv.most_similar(target_word, topn=3)
    print(f"✅ 子任务3完成：与“{target_word}”最相似的3个词")
    for idx, (word, sim) in enumerate(similar_words, 1):
        print(f"{idx}. {word} —— 相似度：{sim:.4f}")
else:
    print(f"⚠️  语料中无“{target_word}”一词")

✅ 子任务3完成：与“好吃”最相似的3个词
1. 油腻 —— 相似度：0.6821
2. 好次 —— 相似度：0.6772
3. 粗 —— 相似度：0.6721


In [27]:
w1, w2, w3 = "好吃", "美味", "蟑螂"
if all(word in w2v_model.wv for word in [w1, w2, w3]):
    sim1 = w2v_model.wv.similarity(w1, w2)
    sim2 = w2v_model.wv.similarity(w1, w3)
    print(f"✅ 子任务4完成：词对相似度计算结果（保留4位小数）")
    print(f'"{w1}" & "{w2}" 的相似度：{sim1:.4f}')
    print(f'"{w1}" & "{w3}" 的相似度：{sim2:.4f}')
else:
    missing = [w for w in [w1, w2, w3] if w not in w2v_model.wv]
    print(f"⚠️  语料中缺少词汇：{missing}")

✅ 子任务4完成：词对相似度计算结果（保留4位小数）
"好吃" & "美味" 的相似度：0.6049
"好吃" & "蟑螂" 的相似度：0.2197


In [28]:
pos_words = ["餐厅", "聚会"]
neg_words = ["安静"]
if all(word in w2v_model.wv for word in pos_words + neg_words):
    result = w2v_model.wv.most_similar(
        positive=pos_words,
        negative=neg_words,
        topn=1
    )
    res_word, res_sim = result[0]
    print(f"✅ 子任务5完成：向量运算结果")
    print(f"「{pos_words[0]} + {pos_words[1]} - {neg_words[0]}」→ 最相关词：{res_word}")
    print(f"相似度：{res_sim:.4f}")
else:
    missing = [w for w in pos_words + neg_words if w not in w2v_model.wv]
    print(f"⚠️  语料中缺少词汇：{missing}")

✅ 子任务5完成：向量运算结果
「餐厅 + 聚会 - 安静」→ 最相关词：客户
相似度：0.7378
